# 自建 overlay 第 2 步 —— 我们自己的 bit 也能放出声

**这是给截图用的笔记本。** 和 `board/scripts/verify_own_overlay_audio.py` 做同一件事,
只是搬到浏览器里跑,输出直接显示在页面上 —— 按 `Win + Shift + S` 就能截。

区别在于"用的是谁的 bit":

| | 加载的 bit | 证明的是什么 |
|---|---|---|
| `w2_audio_playback.ipynb` | PYNQ 出厂预编译的 `base.bit` | 板子会出声 |
| **本笔记本** | **我们自己跑 Vivado 出的 `audio.bit`** | **我们的工程会出声** |

**跑之前**:耳机插板子的 `HP OUT` 口。
**怎么跑**:菜单 `Cell` → `Run All`,或者每个格子按 `Shift + Enter`。


In [ ]:
import math, os, resource, struct, time, wave

from pynq import Overlay

# 和 verify_audio_playback.py 用同一套参数,不另立标准
SAMPLE_RATE = 48000          # codec 固定 48 kHz,不要改
TONE_HZ     = 440.0          # A4
DURATION_S  = 3.0
WAV_PATH    = "/home/xilinx/w2_tone440.wav"
VOLUME      = 62             # 62 是允许的最大值(不是 63)

BITFILE     = "audio.bit"    # 我们自己的 bit
EXPECT_ADDR = 0x43C00000     # 和官方 base overlay 对齐的那个地址


def make_tone(path, hz=TONE_HZ, seconds=DURATION_S, rate=SAMPLE_RATE, amp=0.5):
    """写一个干净的 24 bit / 双声道 / 48 kHz 正弦波。

    load() 只吃这个格式 —— 它是按每采样 3 字节解包的。
    """
    peak = int(amp * (2 ** 23 - 1))
    frames = bytearray()
    for i in range(int(rate * seconds)):
        v = int(peak * math.sin(2 * math.pi * hz * i / rate))
        raw = struct.pack("<i", v)[:3]      # 小端 24 bit
        frames += raw + raw                 # 左右声道同一份
    with wave.open(path, "wb") as f:
        f.setnchannels(2); f.setsampwidth(3); f.setframerate(rate)
        f.writeframes(bytes(frames))


def measure(fn, *args):
    """跑一次 fn,返回 (墙钟秒数, 进程 CPU 秒数)。

    必须先取基准再差分 —— ru_utime/ru_stime 是进程累计值,
    直接读会把加载 overlay 的功夫一起算进来。
    """
    r0 = resource.getrusage(resource.RUSAGE_SELF); t0 = time.time()
    fn(*args)
    wall = time.time() - t0
    r1 = resource.getrusage(resource.RUSAGE_SELF)
    return wall, (r1.ru_utime - r0.ru_utime) + (r1.ru_stime - r0.ru_stime)


print("准备完毕。BITFILE =", BITFILE)


In [ ]:
# ---- 1. 加载我们自己的 bit ----
# 和 base.bit 最直观的区别:那个 .bit 是 PYNQ 出厂就编好的,
# 这个是 build_audio.tcl 在我们机器上跑 vivado 出来的。
print("加载 %s ..." % BITFILE)
ol = Overlay(BITFILE)
print("加载成功")

# ip_dict = 这个 bit 里"有哪些可以访问的 IP、各自在哪个地址"
print("ip_dict:")
for name, info in sorted(ol.ip_dict.items()):
    addr = info.get("phys_addr") if isinstance(info, dict) else "?"
    print("    %-24s %s" % (name, addr))


In [ ]:
# ---- 2. 核对音频 IP:在不在、地址对不对 ----
ip = ol.ip_dict.get("audio_codec_ctrl_0")
assert ip is not None, "这个 bit 里没有音频 IP"
addr = ip.get("phys_addr")
print("音频 IP : audio_codec_ctrl_0")
print("基地址  : 0x%08X   (期望 0x%08X)  %s"
      % (addr, EXPECT_ADDR, "对得上" if addr == EXPECT_ADDR else "对不上!"))
assert addr == EXPECT_ADDR, "地址和官方不一致,驱动不能照用"

# ---- 3. 驱动绑定 ----
# AudioADAU1761.bindto = ['xilinx.com:user:audio_codec_ctrl:1.0'],
# PYNQ 按 vlnv 自动绑,所以这里取到的应该已经是驱动实例而不是裸 MMIO。
audio = ol.audio_codec_ctrl_0
print("驱动类型: %s" % type(audio).__name__)
assert type(audio).__name__ == "AudioADAU1761"


In [ ]:
# ---- 4. 配 codec ----
# configure() 走 PS 侧 I2C1(经 EMIO 到 PL 的 U9/T9),按名字去 /dev 找
# 那个叫 audio-codec-ctrl 的 UIO 节点 —— 这也是基地址必须对齐官方的第二个原因。
audio.configure()
audio.set_volume(VOLUME)
print("configure() 完成,音量设为 %d" % VOLUME)


In [ ]:
# ---- 5. 出声 ----
if not os.path.exists(WAV_PATH):
    print("生成 %s ..." % WAV_PATH)
    make_tone(WAV_PATH)

audio.load(WAV_PATH)          # 把样本灌进 IP 自己的缓冲区
print("播放 %g 秒 %g Hz 正弦波 —— 听!" % (DURATION_S, TONE_HZ))
wall, cpu = measure(audio.play)
print("  墙钟 %.2f 秒   CPU %.2f 秒" % (wall, cpu))


## 判读

看两件事:

**一、地址**

```
基地址  : 0x43C00000   (期望 0x43C00000)  对得上
```

`0x43C00000` 是**我们故意选的** —— 和官方 base overlay 给这个 IP 的地址一模一样。
所以 `pynq.lib.audio` 里写死的寄存器偏移不用改,板子设备树里那个
`audio-codec-ctrl` 的 UIO 节点也照样找得到。**换 bit 不换代码**,就是靠这个。

**二、CPU 时间**

```
播放 3 秒  →  墙钟 2.96 秒   CPU 2.96 秒
```

CPU 时间**等于**墙钟时间 = 一个核被占满了。芯片是双核,所以这等于**整整一个核**。

这个数**和 9/27 用 `base.bit` 跑出来的一模一样** —— 说明换成我们自己造的 bit 之后,
底下走的路完全没变。它不是巧合,是"基地址对齐"这个设计决定的。

**三、还有一条只能靠耳朵**

上面这些检查都过了,也不代表真的出声。**听到那声"嘟"才算过。**
